In [2]:
import numpy as np
import pandas as pd
from pathlib import Path
import plotly.graph_objects as go
from statsmodels.tsa.api import VAR

import ipywidgets as widgets
from IPython.display import display, clear_output

# ============================================================
# TVP-VAR IRF 3D Surface + Event Mapping
# - Input: ./tvpvar_input_scaled.csv
# - Interactive selection: shock / response feature
# - x-axis = Date
# - y-axis = Horizon
# - z-axis = IRF
# ============================================================

# -----------------------------
# Config
# -----------------------------
INPUT_FILE = Path("./tvpvar_input_scaled.csv")
DATE_COL = "Date"

VAR_COLS = [
    "dlog_SOLVPN",
    "dlog_COPPER",
    "dlog_GOLD",
    "dlog_SILVER",
    "dlog_DXY",
    "d_UST10Y",
    "d_VIX"
]

DEFAULT_SHOCK_VAR = "dlog_SOLVPN"
DEFAULT_RESPONSE_VAR = "dlog_COPPER"

LAG_ORDER = 1
ROLLING_WINDOW = 120
HORIZON = 20

MIN_REQUIRED_ROWS = ROLLING_WINDOW + LAG_ORDER + 5

# -----------------------------
# Event Mapping
# -----------------------------
EVENTS = [
    {"date": "2022-11-30", "label": "ChatGPT 출시", "group": "AI / Data Center", "color": "blue"},
    {"date": "2023-01-23", "label": "MSFT–OpenAI 투자/협력", "group": "AI / Data Center", "color": "blue"},
    {"date": "2023-03-14", "label": "GPT-4 공개", "group": "AI / Data Center", "color": "blue"},
    {"date": "2023-11-06", "label": "OpenAI DevDay", "group": "AI / Data Center", "color": "blue"},
    {"date": "2024-01-01", "label": "AI 투자 사이클 시작", "group": "AI / Data Center", "color": "blue"},
    {"date": "2024-03-18", "label": "NVIDIA Blackwell 발표", "group": "AI / Data Center", "color": "blue"},
    {"date": "2024-05-01", "label": "Hyperscaler capex 확대", "group": "AI / Data Center", "color": "blue"},
    {"date": "2024-07-31", "label": "BigTech AI capex 급증", "group": "AI / Data Center", "color": "blue"},
    {"date": "2024-10-15", "label": "OCP Blackwell 관련", "group": "AI / Data Center", "color": "blue"},
    {"date": "2025-01-21", "label": "Stargate 프로젝트", "group": "AI / Data Center", "color": "blue"},

    {"date": "2021-11-15", "label": "구리 공급 부족 신호", "group": "Copper / Supply", "color": "red"},
    {"date": "2023-03-01", "label": "구리 가격 상승 전망", "group": "Copper / Supply", "color": "red"},
    {"date": "2023-11-28", "label": "파나마 구리 광산 폐쇄", "group": "Copper / Supply", "color": "red"},
    {"date": "2024-08-15", "label": "LME 구리 재고 급감", "group": "Copper / Supply", "color": "red"},
    {"date": "2025-02-25", "label": "미국 구리 수입 조사 시작", "group": "Copper / Supply", "color": "red"},
    {"date": "2025-03-10", "label": "구리 정책 리스크 확대", "group": "Copper / Supply", "color": "red"},

    {"date": "2021-04-01", "label": "클라우드 수요 증가", "group": "Infra / System", "color": "green"},
    {"date": "2021-10-28", "label": "Meta 사명 변경", "group": "Infra / System", "color": "green"},
    {"date": "2022-03-22", "label": "H100 발표", "group": "Infra / System", "color": "green"},
    {"date": "2022-06-15", "label": "공급망 불안", "group": "Infra / System", "color": "green"},
    {"date": "2022-09-15", "label": "에너지 가격 급등", "group": "Infra / System", "color": "green"},
    {"date": "2022-09-20", "label": "H100 생산 시작", "group": "Infra / System", "color": "green"},
    {"date": "2023-05-01", "label": "AI 인프라 수요 증가", "group": "Infra / System", "color": "green"},
    {"date": "2023-07-01", "label": "GPU 부족", "group": "Infra / System", "color": "green"},
    {"date": "2023-09-15", "label": "데이터센터 딜 최대치", "group": "Infra / System", "color": "green"},
    {"date": "2023-10-15", "label": "액체 냉각 도입 논의", "group": "Infra / System", "color": "green"},
    {"date": "2024-09-20", "label": "전력 계약 이슈", "group": "Infra / System", "color": "green"},
]

# -----------------------------
# Helper
# -----------------------------
def compute_orth_irf_matrix(var_result, horizon: int) -> np.ndarray:
    irf_obj = var_result.irf(horizon)
    return irf_obj.orth_irfs  # shape: (horizon+1, k, k)

def build_irf_surface_figure(df, shock_var: str, response_var: str):
    shock_idx = VAR_COLS.index(shock_var)
    response_idx = VAR_COLS.index(response_var)

    dates_for_plot = []
    surface_rows = []

    for t in range(ROLLING_WINDOW, len(df)):
        window_df = df.iloc[t - ROLLING_WINDOW:t].copy()
        end_date = df.iloc[t][DATE_COL]
        y = window_df[VAR_COLS].copy()

        try:
            model = VAR(y)
            result = model.fit(LAG_ORDER)

            irf_mat = compute_orth_irf_matrix(result, HORIZON)
            pair_irf = irf_mat[:, response_idx, shock_idx]  # [horizon+1]

            surface_rows.append(pair_irf)
            dates_for_plot.append(end_date)

        except Exception:
            surface_rows.append(np.full(HORIZON + 1, np.nan))
            dates_for_plot.append(end_date)

    surface = np.array(surface_rows)  # [time, horizon+1]

    if surface.size == 0:
        raise ValueError("IRF surface가 생성되지 않았습니다.")

    plot_dates = pd.to_datetime(dates_for_plot)
    horizons = np.arange(HORIZON + 1)

    z = surface.T  # [horizon+1, time]

    z_min = np.nanmin(z)
    z_max = np.nanmax(z)

    plot_date_min = plot_dates.min()
    plot_date_max = plot_dates.max()

    fig = go.Figure()

    fig.add_trace(
        go.Surface(
            x=plot_dates,
            y=horizons,
            z=z,
            colorbar=dict(title="IRF"),
            name="IRF Surface",
            showscale=True
        )
    )

    for event in EVENTS:
        event_date = pd.to_datetime(event["date"])

        if not (plot_date_min <= event_date <= plot_date_max):
            continue

        fig.add_trace(
            go.Scatter3d(
                x=[event_date, event_date],
                y=[0, 0],
                z=[z_min, z_max],
                mode="lines+text",
                line=dict(color=event["color"], width=6),
                text=["", event["label"]],
                textposition="top center",
                name=event["group"],
                hovertemplate=(
                    f"Group: {event['group']}<br>"
                    f"Date: {event_date.strftime('%Y-%m-%d')}<br>"
                    f"Event: {event['label']}<extra></extra>"
                ),
                showlegend=False
            )
        )

    legend_groups = [
        ("AI / Data Center", "blue"),
        ("Copper / Supply", "red"),
        ("Infra / System", "green"),
    ]

    for group_name, color in legend_groups:
        fig.add_trace(
            go.Scatter3d(
                x=[None],
                y=[None],
                z=[None],
                mode="lines",
                line=dict(color=color, width=8),
                name=group_name,
                showlegend=True
            )
        )

    fig.update_layout(
        title=f"TVP-VAR IRF Surface: {shock_var} shock → {response_var} response",
        scene=dict(
            xaxis=dict(
                title="Date",
                autorange="reversed"
            ),
            yaxis=dict(
                title="Horizon"
            ),
            zaxis=dict(
                title="IRF"
            ),
            camera=dict(
                eye=dict(x=1.55, y=1.45, z=0.95)
            )
        ),
        width=1400,
        height=900,
        margin=dict(l=20, r=20, t=60, b=20)
    )

    return fig

# -----------------------------
# Load
# -----------------------------
df = pd.read_csv(INPUT_FILE)
df[DATE_COL] = pd.to_datetime(df[DATE_COL])
df = df.sort_values(DATE_COL).reset_index(drop=True)

use_cols = [DATE_COL] + VAR_COLS
df = df[use_cols].copy()
df = df.dropna().reset_index(drop=True)

if len(df) < MIN_REQUIRED_ROWS:
    raise ValueError(
        f"데이터가 너무 적습니다. 현재 행 수={len(df)}, "
        f"최소 필요 행 수는 대략 {MIN_REQUIRED_ROWS} 이상입니다."
    )

# -----------------------------
# Interactive widgets
# -----------------------------
shock_dropdown = widgets.Dropdown(
    options=VAR_COLS,
    value=DEFAULT_SHOCK_VAR,
    description="Shock:",
    layout=widgets.Layout(width="350px")
)

response_dropdown = widgets.Dropdown(
    options=VAR_COLS,
    value=DEFAULT_RESPONSE_VAR,
    description="Response:",
    layout=widgets.Layout(width="350px")
)

output = widgets.Output()

def update_plot(change=None):
    with output:
        clear_output(wait=True)

        shock_var = shock_dropdown.value
        response_var = response_dropdown.value

        if shock_var == response_var:
            print("Shock 변수와 Response 변수는 서로 달라야 합니다.")
            return

        fig = build_irf_surface_figure(df, shock_var, response_var)
        fig.show()

shock_dropdown.observe(update_plot, names="value")
response_dropdown.observe(update_plot, names="value")

display(widgets.HBox([shock_dropdown, response_dropdown]))
display(output)

update_plot()

Output()

In [1]:
import time
import numpy as np
import pandas as pd
from pathlib import Path
from statsmodels.tsa.api import VAR

# ============================================================
# Rolling VAR IRF (All Pairs) + Monte Carlo CI + Sequential Significance
# ------------------------------------------------------------
# Input:
#   ./tvpvar_input_scaled.csv
#
# Output:
#   1) ./irf_pairwise_sequential_sig_table.csv
#   2) ./irf_pairwise_sequential_summary.csv
#   3) ./irf_pairwise_event_summary.csv
#   4) ./irf_pairwise_debug_table.csv
#   5) ./irf_sanity_sample_table.csv
#
# Core rules:
#   - Use orthogonal IRF
#   - Use errband_mc(..., orth=True)
#   - Sequential significance:
#       if horizon h becomes non-significant,
#       stop that pair/date at h-1
#   - Analyze all ordered pairs (shock != response)
#
# Notes:
#   - This is computationally heavy
#   - Start with MC_REPL=100 for test
#   - Then move to MC_REPL=300 or 500
# ============================================================

# -----------------------------
# Config
# -----------------------------
BASE_DIR = Path("./")
INPUT_FILE = BASE_DIR / "tvpvar_input_scaled.csv"

DATE_COL = "Date"

VAR_COLS = [
    "dlog_SOLVPN",
    "dlog_COPPER",
    "dlog_GOLD",
    "dlog_SILVER",
    "dlog_DXY",
    "d_UST10Y",
    "d_VIX"
]

# Rolling / VAR / IRF
LAG_ORDER_MODE = "fixed"     # "fixed" or "ic"
FIXED_LAG_ORDER = 1
IC_MAXLAGS = 5
IC_CRITERION = "aic"         # only used when LAG_ORDER_MODE == "ic"

ROLLING_WINDOW = 120
HORIZON = 10                 # 먼저 10으로 테스트 권장
SIGNIF = 0.05                # 95% CI
MC_REPL = 100                # 테스트: 100 / 일반: 300~500
MC_BURN = 100
MC_SEED = 42

# Execution controls
RUN_SANITY_EXPORT = True
SANITY_EXPORT_MAX_ROWS = 3000     # sanity sample csv에 저장할 최대 row 수
SAVE_ONLY_HAS_SIG_EVENT = False   # True면 event summary에서 Has_Sig_Path=Yes만 남김

# Logging
LOG_INTERVAL_DATE = 10            # 날짜 step log 주기
LOG_INTERVAL_PAIR = 10            # pair loop log 주기

# Outputs
OUT_SEQ_TABLE = BASE_DIR / "irf_pairwise_sequential_sig_table.csv"
OUT_SUMMARY = BASE_DIR / "irf_pairwise_sequential_summary.csv"
OUT_EVENT_SUMMARY = BASE_DIR / "irf_pairwise_event_summary.csv"
OUT_DEBUG = BASE_DIR / "irf_pairwise_debug_table.csv"
OUT_SANITY = BASE_DIR / "irf_sanity_sample_table.csv"

MIN_REQUIRED_ROWS = ROLLING_WINDOW + max(FIXED_LAG_ORDER, 1) + 5

# -----------------------------
# Event Mapping
# -----------------------------
EVENTS = [
    {"date": "2022-11-30", "label": "ChatGPT 출시", "group": "AI / Data Center"},
    {"date": "2023-01-23", "label": "MSFT–OpenAI 투자/협력", "group": "AI / Data Center"},
    {"date": "2023-03-14", "label": "GPT-4 공개", "group": "AI / Data Center"},
    {"date": "2023-11-06", "label": "OpenAI DevDay", "group": "AI / Data Center"},
    {"date": "2024-01-01", "label": "AI 투자 사이클 시작", "group": "AI / Data Center"},
    {"date": "2024-03-18", "label": "NVIDIA Blackwell 발표", "group": "AI / Data Center"},
    {"date": "2024-05-01", "label": "Hyperscaler capex 확대", "group": "AI / Data Center"},
    {"date": "2024-07-31", "label": "BigTech AI capex 급증", "group": "AI / Data Center"},
    {"date": "2024-10-15", "label": "OCP Blackwell 관련", "group": "AI / Data Center"},
    {"date": "2025-01-21", "label": "Stargate 프로젝트", "group": "AI / Data Center"},

    {"date": "2021-11-15", "label": "구리 공급 부족 신호", "group": "Copper / Supply"},
    {"date": "2023-03-01", "label": "구리 가격 상승 전망", "group": "Copper / Supply"},
    {"date": "2023-11-28", "label": "파나마 구리 광산 폐쇄", "group": "Copper / Supply"},
    {"date": "2024-08-15", "label": "LME 구리 재고 급감", "group": "Copper / Supply"},
    {"date": "2025-02-25", "label": "미국 구리 수입 조사 시작", "group": "Copper / Supply"},
    {"date": "2025-03-10", "label": "구리 정책 리스크 확대", "group": "Copper / Supply"},

    {"date": "2021-04-01", "label": "클라우드 수요 증가", "group": "Infra / System"},
    {"date": "2021-10-28", "label": "Meta 사명 변경", "group": "Infra / System"},
    {"date": "2022-03-22", "label": "H100 발표", "group": "Infra / System"},
    {"date": "2022-06-15", "label": "공급망 불안", "group": "Infra / System"},
    {"date": "2022-09-15", "label": "에너지 가격 급등", "group": "Infra / System"},
    {"date": "2022-09-20", "label": "H100 생산 시작", "group": "Infra / System"},
    {"date": "2023-05-01", "label": "AI 인프라 수요 증가", "group": "Infra / System"},
    {"date": "2023-07-01", "label": "GPU 부족", "group": "Infra / System"},
    {"date": "2023-09-15", "label": "데이터센터 딜 최대치", "group": "Infra / System"},
    {"date": "2023-10-15", "label": "액체 냉각 도입 논의", "group": "Infra / System"},
    {"date": "2024-09-20", "label": "전력 계약 이슈", "group": "Infra / System"},
]

# -----------------------------
# Helpers
# -----------------------------
def format_eta(seconds: float) -> str:
    if seconds is None or np.isnan(seconds):
        return "??:??:??"
    seconds = max(0, int(seconds))
    m, s = divmod(seconds, 60)
    h, m = divmod(m, 60)
    return f"{h:02d}:{m:02d}:{s:02d}"


def is_significant(lower: float, upper: float) -> bool:
    return (lower > 0.0) or (upper < 0.0)


def classify_direction_from_ci(lower: float, upper: float) -> str:
    if lower > 0.0:
        return "Positive"
    if upper < 0.0:
        return "Negative"
    return "None"


def safe_irf_errband_mc(irf_obj, orth: bool, repl: int, signif: float, seed: int, burn: int):
    """
    statsmodels 버전 차이 대응
    """
    errors = []

    try:
        return irf_obj.errband_mc(
            orth=orth,
            repl=repl,
            signif=signif,
            seed=seed,
            burn=burn
        )
    except Exception as e:
        errors.append(f"case1: {repr(e)}")

    try:
        return irf_obj.errband_mc(
            orth=orth,
            repl=repl,
            signif=signif,
            seed=seed
        )
    except Exception as e:
        errors.append(f"case2: {repr(e)}")

    try:
        return irf_obj.errband_mc(
            orth=orth,
            repl=repl,
            signif=signif
        )
    except Exception as e:
        errors.append(f"case3: {repr(e)}")

    raise RuntimeError("errband_mc 호출 실패:\n" + "\n".join(errors))


def fit_var_model(y: pd.DataFrame):
    model = VAR(y)
    if LAG_ORDER_MODE == "fixed":
        return model.fit(FIXED_LAG_ORDER)
    elif LAG_ORDER_MODE == "ic":
        return model.fit(maxlags=IC_MAXLAGS, ic=IC_CRITERION)
    else:
        raise ValueError("LAG_ORDER_MODE must be 'fixed' or 'ic'")


def build_all_pairs(var_cols):
    pairs = []
    for shock_var in var_cols:
        for response_var in var_cols:
            if shock_var == response_var:
                continue
            pairs.append((shock_var, response_var))
    return pairs


# -----------------------------
# Main builder
# -----------------------------
def build_pairwise_irf_tables(df: pd.DataFrame):
    pairs = build_all_pairs(VAR_COLS)
    n_pairs = len(pairs)

    sequential_rows = []
    summary_rows = []
    debug_rows = []
    sanity_rows = []

    total_dates = len(df) - ROLLING_WINDOW
    total_steps = total_dates * n_pairs
    global_start = time.time()
    step_counter = 0

    print("=" * 90)
    print("Start Rolling Pairwise IRF Build")
    print(f"Input file       : {INPUT_FILE}")
    print(f"Rows             : {len(df)}")
    print(f"Variables        : {len(VAR_COLS)}")
    print(f"Ordered pairs    : {n_pairs}")
    print(f"Rolling window   : {ROLLING_WINDOW}")
    print(f"Horizon          : {HORIZON}")
    print(f"Lag mode         : {LAG_ORDER_MODE}")
    if LAG_ORDER_MODE == "fixed":
        print(f"Fixed lag        : {FIXED_LAG_ORDER}")
    else:
        print(f"IC maxlags       : {IC_MAXLAGS}")
        print(f"IC criterion     : {IC_CRITERION}")
    print(f"MC_REPL          : {MC_REPL}")
    print(f"SIGNIF           : {SIGNIF}")
    print("=" * 90)

    for t in range(ROLLING_WINDOW, len(df)):
        date_step = t - ROLLING_WINDOW + 1
        end_date = pd.to_datetime(df.iloc[t][DATE_COL])

        if date_step == 1 or date_step % LOG_INTERVAL_DATE == 0 or date_step == total_dates:
            elapsed = time.time() - global_start
            approx_progress = step_counter / total_steps if total_steps > 0 else 0.0
            eta = (elapsed / approx_progress - elapsed) if approx_progress > 0 else np.nan
            print(
                f"[DATE {date_step:>4}/{total_dates}] "
                f"Date={end_date.date()} | "
                f"Elapsed={format_eta(elapsed)} | ETA={format_eta(eta)}"
            )

        window_df = df.iloc[t - ROLLING_WINDOW:t].copy()
        y = window_df[VAR_COLS].copy()

        # --- fit once per date window ---
        try:
            result = fit_var_model(y)
            selected_lag = int(result.k_ar)
            irf_obj = result.irf(HORIZON)

            # IMPORTANT:
            # point estimate = orth_irfs
            # CI = errband_mc(..., orth=True)
            irf_mat = irf_obj.orth_irfs
            lower_mat, upper_mat = safe_irf_errband_mc(
                irf_obj=irf_obj,
                orth=True,
                repl=MC_REPL,
                signif=SIGNIF,
                seed=MC_SEED,
                burn=MC_BURN
            )

            model_error = ""
        except Exception as e:
            result = None
            selected_lag = np.nan
            irf_mat = None
            lower_mat = None
            upper_mat = None
            model_error = str(e)

        for pair_idx, (shock_var, response_var) in enumerate(pairs, start=1):
            step_counter += 1

            if pair_idx == 1 or pair_idx % LOG_INTERVAL_PAIR == 0 or pair_idx == n_pairs:
                elapsed = time.time() - global_start
                progress = step_counter / total_steps if total_steps > 0 else 0.0
                eta = (elapsed / progress - elapsed) if progress > 0 else np.nan
                print(
                    f"    [PAIR {pair_idx:>2}/{n_pairs}] "
                    f"{shock_var} -> {response_var} | "
                    f"Progress={progress*100:6.2f}% | ETA={format_eta(eta)}"
                )

            shock_idx = VAR_COLS.index(shock_var)
            response_idx = VAR_COLS.index(response_var)

            summary_record = {
                "Date": end_date,
                "Shock_Var": shock_var,
                "Response_Var": response_var,
                "Selected_Lag": selected_lag,
                "Has_Sig_Path": "No",
                "Start_H": np.nan,
                "End_H": np.nan,
                "Sig_Count": 0,
                "Peak_IRF": np.nan,
                "Peak_Abs_IRF": np.nan,
                "Peak_H": np.nan,
                "Cum_IRF": 0.0,
                "Mean_IRF": np.nan,
                "Direction": "None",
                "H0_IRF": np.nan,
                "H0_CI_Lower": np.nan,
                "H0_CI_Upper": np.nan,
                "H0_Significant": "No",
                "Stop_H": np.nan,
                "Stop_Reason": "",
                "Model_Error": model_error,
            }

            debug_record = {
                "Date": end_date,
                "Shock_Var": shock_var,
                "Response_Var": response_var,
                "Selected_Lag": selected_lag,
                "H0_IRF": np.nan,
                "H0_CI_Lower": np.nan,
                "H0_CI_Upper": np.nan,
                "H0_Significant": "No",
                "H0_Direction": "None",
                "Stop_H": np.nan,
                "Stop_Reason": "",
                "Model_Error": model_error,
            }

            if irf_mat is None:
                summary_record["Stop_Reason"] = "Model fit / IRF error"
                debug_record["Stop_Reason"] = "Model fit / IRF error"
                summary_rows.append(summary_record)
                debug_rows.append(debug_record)
                continue

            # h=0 always record in summary/debug
            h0_irf = float(irf_mat[0, response_idx, shock_idx])
            h0_lower = float(lower_mat[0, response_idx, shock_idx])
            h0_upper = float(upper_mat[0, response_idx, shock_idx])
            h0_sig = is_significant(h0_lower, h0_upper)
            h0_dir = classify_direction_from_ci(h0_lower, h0_upper)

            summary_record["H0_IRF"] = h0_irf
            summary_record["H0_CI_Lower"] = h0_lower
            summary_record["H0_CI_Upper"] = h0_upper
            summary_record["H0_Significant"] = "Yes" if h0_sig else "No"

            debug_record["H0_IRF"] = h0_irf
            debug_record["H0_CI_Lower"] = h0_lower
            debug_record["H0_CI_Upper"] = h0_upper
            debug_record["H0_Significant"] = "Yes" if h0_sig else "No"
            debug_record["H0_Direction"] = h0_dir

            if RUN_SANITY_EXPORT and len(sanity_rows) < SANITY_EXPORT_MAX_ROWS:
                for h in range(HORIZON + 1):
                    sanity_irf = float(irf_mat[h, response_idx, shock_idx])
                    sanity_lower = float(lower_mat[h, response_idx, shock_idx])
                    sanity_upper = float(upper_mat[h, response_idx, shock_idx])
                    sanity_sig = "Yes" if is_significant(sanity_lower, sanity_upper) else "No"
                    sanity_rows.append({
                        "Date": end_date,
                        "Shock_Var": shock_var,
                        "Response_Var": response_var,
                        "Horizon": h,
                        "IRF": sanity_irf,
                        "CI_Lower": sanity_lower,
                        "CI_Upper": sanity_upper,
                        "Significant": sanity_sig,
                    })
                    if len(sanity_rows) >= SANITY_EXPORT_MAX_ROWS:
                        break

            valid_irfs = []
            valid_dirs = []

            # sequential significance
            for h in range(HORIZON + 1):
                irf_val = float(irf_mat[h, response_idx, shock_idx])
                ci_lower = float(lower_mat[h, response_idx, shock_idx])
                ci_upper = float(upper_mat[h, response_idx, shock_idx])

                sig = is_significant(ci_lower, ci_upper)

                if not sig:
                    summary_record["Stop_H"] = h
                    summary_record["Stop_Reason"] = "Non-significant CI"
                    debug_record["Stop_H"] = h
                    debug_record["Stop_Reason"] = "Non-significant CI"
                    break

                direction = classify_direction_from_ci(ci_lower, ci_upper)

                sequential_rows.append({
                    "Date": end_date,
                    "Shock_Var": shock_var,
                    "Response_Var": response_var,
                    "Selected_Lag": selected_lag,
                    "Horizon": h,
                    "IRF": irf_val,
                    "CI_Lower": ci_lower,
                    "CI_Upper": ci_upper,
                    "Significant": "Yes",
                    "Direction": direction,
                })

                valid_irfs.append(irf_val)
                valid_dirs.append(direction)

            else:
                summary_record["Stop_H"] = HORIZON
                summary_record["Stop_Reason"] = "Reached max horizon"
                debug_record["Stop_H"] = HORIZON
                debug_record["Stop_Reason"] = "Reached max horizon"

            if len(valid_irfs) > 0:
                valid_irfs_np = np.array(valid_irfs, dtype=float)
                peak_idx = int(np.argmax(np.abs(valid_irfs_np)))
                peak_irf = float(valid_irfs_np[peak_idx])
                peak_abs_irf = float(np.abs(valid_irfs_np[peak_idx]))
                peak_h = peak_idx

                uniq_dirs = sorted(set(valid_dirs))
                main_direction = uniq_dirs[0] if len(uniq_dirs) == 1 else "Mixed"

                summary_record.update({
                    "Has_Sig_Path": "Yes",
                    "Start_H": 0,
                    "End_H": len(valid_irfs) - 1,
                    "Sig_Count": len(valid_irfs),
                    "Peak_IRF": peak_irf,
                    "Peak_Abs_IRF": peak_abs_irf,
                    "Peak_H": peak_h,
                    "Cum_IRF": float(valid_irfs_np.sum()),
                    "Mean_IRF": float(valid_irfs_np.mean()),
                    "Direction": main_direction,
                })

            summary_rows.append(summary_record)
            debug_rows.append(debug_record)

    sequential_df = pd.DataFrame(sequential_rows)
    summary_df = pd.DataFrame(summary_rows)
    debug_df = pd.DataFrame(debug_rows)
    sanity_df = pd.DataFrame(sanity_rows)

    if not sequential_df.empty:
        sequential_df["Date"] = pd.to_datetime(sequential_df["Date"])
        sequential_df = sequential_df.sort_values(
            ["Date", "Shock_Var", "Response_Var", "Horizon"]
        ).reset_index(drop=True)

    if not summary_df.empty:
        summary_df["Date"] = pd.to_datetime(summary_df["Date"])
        summary_df = summary_df.sort_values(
            ["Date", "Shock_Var", "Response_Var"]
        ).reset_index(drop=True)

    if not debug_df.empty:
        debug_df["Date"] = pd.to_datetime(debug_df["Date"])
        debug_df = debug_df.sort_values(
            ["Date", "Shock_Var", "Response_Var"]
        ).reset_index(drop=True)

    if not sanity_df.empty:
        sanity_df["Date"] = pd.to_datetime(sanity_df["Date"])
        sanity_df = sanity_df.sort_values(
            ["Date", "Shock_Var", "Response_Var", "Horizon"]
        ).reset_index(drop=True)

    return sequential_df, summary_df, debug_df, sanity_df


# -----------------------------
# Event summary
# -----------------------------
def build_pairwise_event_summary(summary_df: pd.DataFrame):
    if summary_df.empty:
        return pd.DataFrame(columns=[
            "Event_Date", "Event_Label", "Event_Group", "Matched_Date",
            "Shock_Var", "Response_Var", "Selected_Lag", "Has_Sig_Path",
            "End_H", "Sig_Count", "Peak_IRF", "Peak_Abs_IRF", "Peak_H",
            "Cum_IRF", "Direction", "H0_IRF", "H0_Significant"
        ])

    rows = []
    summary_df = summary_df.copy()
    summary_df["Date"] = pd.to_datetime(summary_df["Date"])

    pair_keys = summary_df[["Shock_Var", "Response_Var"]].drop_duplicates()

    for _, pair_row in pair_keys.iterrows():
        shock_var = pair_row["Shock_Var"]
        response_var = pair_row["Response_Var"]

        sub = summary_df[
            (summary_df["Shock_Var"] == shock_var) &
            (summary_df["Response_Var"] == response_var)
        ].copy()

        if sub.empty:
            continue

        for event in EVENTS:
            event_date = pd.to_datetime(event["date"])
            idx = (sub["Date"] - event_date).abs().idxmin()
            row = sub.loc[idx]

            rows.append({
                "Event_Date": event_date,
                "Event_Label": event["label"],
                "Event_Group": event["group"],
                "Matched_Date": row["Date"],
                "Shock_Var": shock_var,
                "Response_Var": response_var,
                "Selected_Lag": row["Selected_Lag"],
                "Has_Sig_Path": row["Has_Sig_Path"],
                "End_H": row["End_H"],
                "Sig_Count": row["Sig_Count"],
                "Peak_IRF": row["Peak_IRF"],
                "Peak_Abs_IRF": row["Peak_Abs_IRF"],
                "Peak_H": row["Peak_H"],
                "Cum_IRF": row["Cum_IRF"],
                "Direction": row["Direction"],
                "H0_IRF": row["H0_IRF"],
                "H0_Significant": row["H0_Significant"],
            })

    event_df = pd.DataFrame(rows)

    if not event_df.empty:
        event_df["Event_Date"] = pd.to_datetime(event_df["Event_Date"])
        event_df["Matched_Date"] = pd.to_datetime(event_df["Matched_Date"])
        event_df = event_df.sort_values(
            ["Shock_Var", "Response_Var", "Event_Date"]
        ).reset_index(drop=True)

        if SAVE_ONLY_HAS_SIG_EVENT:
            event_df = event_df[event_df["Has_Sig_Path"] == "Yes"].reset_index(drop=True)

    return event_df


# -----------------------------
# Load
# -----------------------------
df = pd.read_csv(INPUT_FILE)
df[DATE_COL] = pd.to_datetime(df[DATE_COL])
df = df.sort_values(DATE_COL).reset_index(drop=True)

use_cols = [DATE_COL] + VAR_COLS
df = df[use_cols].copy()
df = df.dropna().reset_index(drop=True)

if len(df) < MIN_REQUIRED_ROWS:
    raise ValueError(
        f"데이터가 너무 적습니다. 현재 행 수={len(df)}, "
        f"최소 필요 행 수는 대략 {MIN_REQUIRED_ROWS} 이상입니다."
    )

# -----------------------------
# Run
# -----------------------------
sequential_df, summary_df, debug_df, sanity_df = build_pairwise_irf_tables(df)
event_summary_df = build_pairwise_event_summary(summary_df)

# -----------------------------
# Save
# -----------------------------
sequential_df.to_csv(OUT_SEQ_TABLE, index=False, encoding="utf-8-sig")
summary_df.to_csv(OUT_SUMMARY, index=False, encoding="utf-8-sig")
event_summary_df.to_csv(OUT_EVENT_SUMMARY, index=False, encoding="utf-8-sig")
debug_df.to_csv(OUT_DEBUG, index=False, encoding="utf-8-sig")
sanity_df.to_csv(OUT_SANITY, index=False, encoding="utf-8-sig")

# -----------------------------
# Post summary
# -----------------------------
print("\n" + "=" * 90)
print("Saved files")
print(f"1) {OUT_SEQ_TABLE}")
print(f"2) {OUT_SUMMARY}")
print(f"3) {OUT_EVENT_SUMMARY}")
print(f"4) {OUT_DEBUG}")
print(f"5) {OUT_SANITY}")
print("=" * 90)

print("\n[Row counts]")
print(f"Sequential rows : {len(sequential_df)}")
print(f"Summary rows    : {len(summary_df)}")
print(f"Event rows      : {len(event_summary_df)}")
print(f"Debug rows      : {len(debug_df)}")
print(f"Sanity rows     : {len(sanity_df)}")

if not summary_df.empty:
    print("\n[Has_Sig_Path counts]")
    print(summary_df["Has_Sig_Path"].value_counts(dropna=False))

    print("\n[Stop_Reason counts]")
    print(summary_df["Stop_Reason"].value_counts(dropna=False).head(10))

    print("\n[Summary preview]")
    print(summary_df.head(10))

if not sanity_df.empty:
    print("\n[Sanity preview]")
    print(sanity_df.head(20))

Start Rolling Pairwise IRF Build
Input file       : tvpvar_input_scaled.csv
Rows             : 1036
Variables        : 7
Ordered pairs    : 42
Rolling window   : 120
Horizon          : 10
Lag mode         : fixed
Fixed lag        : 1
MC_REPL          : 100
SIGNIF           : 0.05
[DATE    1/916] Date=2021-05-20 | Elapsed=00:00:00 | ETA=??:??:??
    [PAIR  1/42] dlog_SOLVPN -> dlog_COPPER | Progress=  0.00% | ETA=02:14:02
    [PAIR 10/42] dlog_COPPER -> dlog_DXY | Progress=  0.03% | ETA=00:13:27
    [PAIR 20/42] dlog_SILVER -> dlog_COPPER | Progress=  0.05% | ETA=00:06:43
    [PAIR 30/42] dlog_DXY -> d_VIX | Progress=  0.08% | ETA=00:04:29
    [PAIR 40/42] d_VIX -> dlog_SILVER | Progress=  0.10% | ETA=00:03:22
    [PAIR 42/42] d_VIX -> d_UST10Y | Progress=  0.11% | ETA=00:03:13
    [PAIR  1/42] dlog_SOLVPN -> dlog_COPPER | Progress=  0.11% | ETA=00:06:08
    [PAIR 10/42] dlog_COPPER -> dlog_DXY | Progress=  0.14% | ETA=00:05:04
    [PAIR 20/42] dlog_SILVER -> dlog_COPPER | Progress=  0.